# Dataset Description
This project uses the Heart Disease Dataset from Kaggle: https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset

The dataset contains multiple clinical and physiological features related to patients’ heart health. These features are used to predict whether a patient has heart disease. The target variable is binary, where 0 indicates no heart disease and 1 indicates the presence of heart disease.

| Feature                           | Description                                                      | Type / Values                                                                           |
| --------------------------------- | ---------------------------------------------------------------- | --------------------------------------------------------------------------------------- |
| Age                               | The age of the patient                                           | Continuous value                                                                        |
| Sex                               | The gender of the patient                                        | `0` = Female, `1` = Male                                                                |
| Chest Pain Type                   | The type of chest pain experienced by the patient                | `0` = Typical angina, `1` = Atypical angina, `2` = Non-anginal pain, `3` = Asymptomatic |
| Resting Blood Pressure            | Resting blood pressure of the patient                            | Continuous value, measured in mmHg                                                      |
| Cholesterol                       | Serum cholesterol level                                          | Continuous value, measured in mg/dl                                                     |
| Fasting Blood Sugar               | Whether fasting blood sugar is greater than 120 mg/dl            | `1` = Greater than 120 mg/dl, `0` = Less than or equal to 120 mg/dl                     |
| Resting ECG Results               | Resting electrocardiographic results                             | `0` = Normal, `1` = ST-T wave abnormality, `2` = Possible left ventricular hypertrophy  |
| Maximum Heart Rate                | Maximum heart rate achieved                                      | Continuous value                                                                        |
| Exercise-Induced Angina           | Whether the patient has exercise-induced angina                  | `1` = Yes, `0` = No                                                                     |
| ST Depression                     | ST depression induced by exercise relative to rest               | Continuous value                                                                        |
| Slope of Peak Exercise ST Segment | The slope of the peak exercise ST segment                        | `0` = Upsloping, `1` = Flat, `2` = Downsloping                                          |
| Number of Major Vessels           | Number of major vessels colored by fluoroscopy                   | Integer value from `0` to `3`                                                           |
| Thalassemia                       | A blood disorder related to inherited anemia                     | `1` = Normal, `2` = Fixed defect, `3` = Reversible defect                               |
| Heart Disease                     | Target variable indicating whether the patient has heart disease | `0` = No, `1` = Yes                                                                     |


In [1]:
import pandas as pd

# 加载数据集
heart_disease = pd.read_csv("data/heart_disease.csv")
# 处理缺失值
heart_disease.dropna()
heart_disease.info()
heart_disease.head()

<class 'pandas.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   年龄        1025 non-null   int64  
 1   性别        1025 non-null   int64  
 2   胸痛类型      1025 non-null   int64  
 3   静息血压      1025 non-null   int64  
 4   胆固醇       1025 non-null   int64  
 5   空腹血糖      1025 non-null   int64  
 6   静息心电图结果   1025 non-null   int64  
 7   最大心率      1025 non-null   int64  
 8   运动性心绞痛    1025 non-null   int64  
 9   运动后的ST下降  1025 non-null   float64
 10  峰值ST段的斜率  1025 non-null   int64  
 11  主血管数量     1025 non-null   int64  
 12  地中海贫血     1025 non-null   int64  
 13  是否患有心脏病   1025 non-null   int64  
dtypes: float64(1), int64(13)
memory usage: 112.2 KB


,年龄,性别,胸痛类型,静息血压,胆固醇,空腹血糖,静息心电图结果,最大心率,运动性心绞痛,运动后的ST下降,峰值ST段的斜率,主血管数量,地中海贫血,是否患有心脏病
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


# Split Data Set

In [3]:
from sklearn.model_selection import train_test_split

# 划分特征和标签
X = heart_disease.drop("是否患有心脏病", axis=1)  # 特征
y = heart_disease["是否患有心脏病"]  # 标签
# 将数据集按7:3划分为训练数据与测试数据
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=100)

数据集中包含多种类型的特征：
- 类别型特征（需要特殊处理）
  - 胸痛类型：4种分类（名义变量）
  - 静息心电图结果：3种分类（名义变量）
  - 峰值ST段的斜率：3种分类（有序变量）
  - 地中海贫血：3种分类（名义变量）
  - 数值型特征（可直接标准化）：年龄、静息血压、胆固醇、最大心率、运动后的ST下降、主血管数量
- 二元特征（保持原样）：性别、空腹血糖、运动性心绞痛
-
对于类别型特征，直接使用整数编码的类别特征会被算法视为有序数值，导致错误的距离计算（例如：会认为 胸痛类型=1 和 胸痛类型=2 之间的差异比 胸痛类型=1和 胸痛类型=3之间差异更小，而实际上它们都是类别）。使用 独热编码（One-Hot Encoding）可将类别特征转换为二元向量，消除虚假的顺序关系。

In [10]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# 数值型特征
numerical_features = ["年龄", "静息血压", "胆固醇", "最大心率", "运动后的ST下降", "主血管数量"]
# 类别型特征
categorical_features = ["胸痛类型", "静息心电图结果", "峰值ST段的斜率", "地中海贫血"]
# 二元特征
binary_features = ["性别", "空腹血糖", "运动性心绞痛"]
# 创建列转换器
preprocessor = ColumnTransformer(
    transformers=[
        # 对数值型特征进行标准化 StandardScaler 会把数据转换成类似下面的形式： 新值 = (原始值 - 平均值) / 标准差
        # 这样做的好处是：
        # 模型训练更稳定，尤其是逻辑回归、SVM、KNN、神经网络这类模型。
        ("num", StandardScaler(), numerical_features),
        # 对类别型特征进行独热编码，使用drop="first"避免多重共线性
        ("cat", OneHotEncoder(drop="first"), categorical_features),
        # 二元特征不进行处理
        ("binary", "passthrough", binary_features),
    ]
)
# 执行特征转换
x_train = preprocessor.fit_transform(x_train)  # 计算训练集的统计信息并进行转换
x_test = preprocessor.transform(x_test)  # 使用训练集计算的信息对测试集进行转换

避免多重共线性
drop="first"是独热编码中的一个参数，它的核心目的是避免多重共线性（Multicollinearity）。
多重共线性是指特征之间存在高度线性相关关系的现象。例如特征胸痛类型包含4个类别（0、1、2、3），若直接进行独热编码会生成4个新列（胸痛类型_0、胸痛类型_1、胸痛类型_2、胸痛类型_3），此时这4列满足
胸痛类型_0+胸痛类型_1+胸痛类型_2+胸痛类型_3=1
这种完全线性相关关系会导致特征矩阵的列之间存在完美共线性。

当特征矩阵存在多重共线性时，模型参数估计会变得不稳定（矩阵不可逆或接近奇异），导致系数估计值方差增大、模型可解释性下降、过拟合等问题。

在独热编码时设置drop="first"，会删除每个类别特征的第1列，从而打破完全共线性。比如特征胸痛类型会生成3列（胸痛类型_1、胸痛类型_2、胸痛类型_3），此时
胸痛类型_1=0，胸痛类型_2=0，胸痛类型_3=0 隐含代表 胸痛类型_0=1

# Model Train and evaluate




In [14]:
from sklearn.neighbors import KNeighborsClassifier

# 使用K近邻分类模型，K=3
knn = KNeighborsClassifier(n_neighbors=3)
# 模型训练
knn.fit(x_train, y_train)
# 模型评估，计算准确率
knn.score(x_test, y_test)

0.9188311688311688

In [15]:
# save model
import joblib

joblib.dump(knn, "knn_heart_disease")

# 加载模型
knn_loaded = joblib.load("knn_heart_disease")

# 预测
y_pred = knn_loaded.predict(x_test[10:11])
# 打印真实值与预测值
print(y_test.iloc[10], y_pred)

0 [0]


# Model Evaluation and HyperParameter tuning
Grid Search is a simple and efficient algorithm for hyperparameter tuning. It works by dividing the hyperparameter space into a grid of points and evaluating the performance of the model on each point. The algorithm then selects the point with the best performance and uses it as the optimal hyperparameter values.


In [16]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV

knn = KNeighborsClassifier()
# 网格搜索参数，K值设置为1到10
param_grid = {"n_neighbors": list(range(1, 10))}
# GridSearchCV(estimator=模型, param_grid=网格搜索参数, cv=k折交叉验证)
knn = GridSearchCV(estimator=knn, param_grid=param_grid, cv=10)

# 模型训练
knn.fit(x_train, y_train)
print(pd.DataFrame(knn.cv_results_))  # 所有交叉验证结果
print(knn.best_estimator_)  # 最佳模型
print(knn.best_score_)  # 最佳得分

# 使用最佳模型进行评估
knn = knn.best_estimator_
print(knn.score(x_test, y_test))

   mean_fit_time  std_fit_time  mean_score_time  std_score_time  \
0       0.000716      0.000088         0.002556        0.000219   
1       0.000708      0.000062         0.002493        0.000180   
2       0.000676      0.000079         0.002453        0.000211   
3       0.000695      0.000084         0.002520        0.000270   
4       0.000718      0.000061         0.002761        0.000261   
5       0.000725      0.000089         0.002702        0.000138   
6       0.000709      0.000035         0.002710        0.000133   
7       0.000721      0.000080         0.002708        0.000143   
8       0.000903      0.000181         0.003144        0.000368   

   param_n_neighbors              params  split0_test_score  \
0                  1  {'n_neighbors': 1}           0.986111   
1                  2  {'n_neighbors': 2}           0.930556   
2                  3  {'n_neighbors': 3}           0.888889   
3                  4  {'n_neighbors': 4}           0.861111   
4             

# Logistic Regression

In [17]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression

# 加载数据集
heart_disease = pd.read_csv("data/heart_disease.csv")
heart_disease.dropna()

# 划分为训练集与测试集
X = heart_disease.drop("是否患有心脏病", axis=1)  # 特征
y = heart_disease["是否患有心脏病"]  # 标签
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=100)

# 特征工程
# 数值型特征
numerical_features = ["年龄", "静息血压", "胆固醇", "最大心率", "运动后的ST下降", "主血管数量"]
# 类别型特征
categorical_features = ["胸痛类型", "静息心电图结果", "峰值ST段的斜率", "地中海贫血"]
# 二元特征
binary_features = ["性别", "空腹血糖", "运动性心绞痛"]

# 创建列转换器
preprocessor = ColumnTransformer(
    transformers=[
        # 对数值型特征进行标准化
        ("num", StandardScaler(), numerical_features),
        # 对类别型特征进行独热编码，使用drop="first"避免多重共线性
        ("cat", OneHotEncoder(drop="first"), categorical_features),
        # 二元特征不进行处理
        ("binary", "passthrough", binary_features),
    ]
)
# 执行特征转换
x_train = preprocessor.fit_transform(x_train)  # 计算训练集的统计信息并进行转换
x_test = preprocessor.transform(x_test)  # 使用训练集计算的信息对测试集进行转换

# 模型训练
model = LogisticRegression()
model.fit(x_train, y_train)

# 模型评估，计算准确率
model.score(x_test, y_test)


0.8441558441558441